# 19 — Paper Figures: t-SNE and KDE of Real vs. Synthetic SEP Samples

For each of the four augmentation algorithms (SMOTE, ADASYN, TimeGAN, Diffusion),
compares the 118 real SEP training sequences against 118 of that algorithm's own
synthetic SEP sequences, in the flattened (288x10 -> 2880-d) feature space.

Row 1: a 2-D t-SNE embedding of real vs. synthetic points, fit jointly per method
(so real and synthetic share one embedding, not two independently-fit ones).
Row 2: a 1-D KDE of each method's real and synthetic points along that same
embedding's first dimension, showing how much the two distributions overlap.

**Reads:** `./sep_samples/sep_real_vs_synthetic_{smote,adasyn,timegan,diffusion}.pkl`
(each holds `X_real_sep`, `X_synthetic_sep`, both shape (118, 288, 10), saved by
notebooks 5/6/8/9 at generation time -- no retraining needed here).
**Requires:** notebook 11 run first (style and loader).
**Writes:** `./paper/figures/augmentation_tsne_kde.pdf` / `.png`.

## 1. Load and Embed

In [3]:
import pickle
from sklearn.manifold import TSNE
from scipy.stats import gaussian_kde

SAMPLES_DIR = "./sep_samples"

AUG_ORDER = ["SMOTE", "ADASYN", "TimeGAN", "Diffusion"]
AUG_FILES = {"SMOTE": "smote", "ADASYN": "adasyn",
             "TimeGAN": "timegan", "Diffusion": "diffusion"}
AUG_FAMILY = {"SMOTE": "classical", "ADASYN": "classical",
              "TimeGAN": "generative", "Diffusion": "generative"}
FAMILY_COLOR = {"classical": TEAL, "generative": VERM}
REAL_COLOR = "0.25"


def load_real_synth(name):
    with open(f"{SAMPLES_DIR}/sep_real_vs_synthetic_{name}.pkl", "rb") as f:
        d = pickle.load(f)
    return d["X_real_sep"], d["X_synthetic_sep"]


EMB = {}  # method -> dict(real_2d, synth_2d)
for a in AUG_ORDER:
    X_real, X_synth = load_real_synth(AUG_FILES[a])
    n_real = X_real.shape[0]
    flat = np.concatenate([X_real, X_synth], axis=0).reshape(n_real + X_synth.shape[0], -1)
    perplexity = min(30, (flat.shape[0] - 1) // 3)
    emb = TSNE(n_components=2, perplexity=perplexity, init="pca",
               random_state=0, learning_rate="auto").fit_transform(flat)
    EMB[a] = {"real": emb[:n_real], "synth": emb[n_real:]}
    print(f"{a:<10} real={X_real.shape} synth={X_synth.shape} "
          f"perplexity={perplexity} embedding={emb.shape}")

SMOTE      real=(118, 288, 10) synth=(118, 288, 10) perplexity=30 embedding=(236, 2)


ADASYN     real=(118, 288, 10) synth=(118, 288, 10) perplexity=30 embedding=(236, 2)


TimeGAN    real=(118, 288, 10) synth=(118, 288, 10) perplexity=30 embedding=(236, 2)


Diffusion  real=(118, 288, 10) synth=(118, 288, 10) perplexity=30 embedding=(236, 2)


## Figure — t-SNE and KDE, Real vs. Synthetic, per Algorithm

In [4]:
# FIG -- t-SNE embeddings (row 1) and KDE along t-SNE dim 1 (row 2),
# real vs synthetic SEP sequences, one column per augmentation algorithm.
from matplotlib.lines import Line2D

fig, axes = plt.subplots(2, 4, figsize=(7.16, 4.6),
                          gridspec_kw=dict(height_ratios=[1.3, 1.0]))

for j, a in enumerate(AUG_ORDER):
    col = FAMILY_COLOR[AUG_FAMILY[a]]
    real2d, synth2d = EMB[a]["real"], EMB[a]["synth"]

    # ---- row 1: t-SNE scatter ----
    ax = axes[0, j]
    ax.scatter(real2d[:, 0], real2d[:, 1], s=7, color=REAL_COLOR, alpha=0.7,
               lw=0, zorder=3, label="real")
    ax.scatter(synth2d[:, 0], synth2d[:, 1], s=7, color=col, alpha=0.6,
               lw=0, zorder=3, label="synthetic")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ("top", "right", "left", "bottom"):
        ax.spines[s].set_visible(False)
    ax.set_title(a, loc="center", fontsize=13.5, pad=6,
                 color=col, fontweight="bold")

    # ---- row 2: KDE along t-SNE dim 1 ----
    ax = axes[1, j]
    all1 = np.concatenate([real2d[:, 0], synth2d[:, 0]])
    xs = np.linspace(all1.min() - 5, all1.max() + 5, 300)
    kde_real = gaussian_kde(real2d[:, 0])(xs)
    kde_synth = gaussian_kde(synth2d[:, 0])(xs)
    ax.fill_between(xs, kde_real, color=REAL_COLOR, alpha=0.25, lw=0, zorder=2)
    ax.plot(xs, kde_real, color=REAL_COLOR, lw=1.0, zorder=3)
    ax.fill_between(xs, kde_synth, color=col, alpha=0.25, lw=0, zorder=2)
    ax.plot(xs, kde_synth, color=col, lw=1.0, zorder=3)
    ax.set_yticks([])
    ax.set_xticks([])
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.set_xlabel("t-SNE dim. 1", fontsize=12.0)

axes[0, 0].set_ylabel("t-SNE dim. 2", fontsize=12.5)
axes[1, 0].set_ylabel("density", fontsize=12.5)

fig.suptitle("Real vs. synthetic SEP sequences, by augmentation algorithm",
             fontsize=15.5, y=0.985)

# One large, centered legend in the empty band between the scatter row and
# the KDE row: real (dark gray), plus synthetic split into its two actual
# panel colors -- green (TEAL) for the classical panels (SMOTE, ADASYN),
# orange (VERM) for the generative panels (TimeGAN, Diffusion) -- rather
# than one generic "synthetic" swatch that wouldn't match either family.
# This legend now carries the classical/generative distinction on its own,
# so the small corner tags that used to duplicate it have been removed.
legend_handles = [
    Line2D([0], [0], marker="o", linestyle="none", color=REAL_COLOR,
           markersize=9, label="real"),
    Line2D([0], [0], marker="o", linestyle="none", color=TEAL,
           markersize=9, label="synthetic (classical)"),
    Line2D([0], [0], marker="o", linestyle="none", color=VERM,
           markersize=9, label="synthetic (generative)"),
]
fig.legend(handles=legend_handles, loc="center", bbox_to_anchor=(0.5, 0.47),
           ncol=3, frameon=False, fontsize=12.0, handletextpad=0.5,
           columnspacing=1.6, markerscale=1.4)

fig.subplots_adjust(left=0.07, right=0.99, top=0.82, bottom=0.10,
                     wspace=0.18, hspace=0.50)
fig.savefig(f"{FIG_DIR}/augmentation_tsne_kde.pdf")
fig.savefig(f"{FIG_DIR}/augmentation_tsne_kde.png", dpi=340)
plt.show()
print("Saved augmentation_tsne_kde.pdf/png")


Saved augmentation_tsne_kde.pdf/png


/var/folders/fx/gjhbmrbj5jn295_9wrqpbsv80000gn/T/ipykernel_85641/3378555978.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
